# scvi/01 — Train SCVI + SCANVI Reference Model

Trains a **SCVI** encoder on reference cells, then wraps it in a **SCANVI**
semi-supervised classifier that learns to predict cell-type labels.
The trained model is saved to disk; query projection (NB scvi/02) loads it
directly — **no retraining needed**.

**scANVI save/load strategy:**
- `model.save(dir, save_anndata=True)` — serialises weights + gene registry + reference AnnData
- `SCANVI.load(dir)` — reload on any machine, apply to new cells via `.get_latent_representation()` + `.predict()`

**Inputs:** `data/scvi/reference.h5ad`  
**Outputs:** `models/scanvi_reference/`  (model), `data/scvi/reference_trained.h5ad`  (with UMAP)

In [1]:
# Parameters — injected by papermill
reference_h5ad  = "data/scvi/reference.h5ad"
model_dir       = "models/scanvi_reference"
labels_key      = "cluster"     # cell-type annotation column from NB00
n_latent        = 30
n_epochs_scvi   = 400
n_epochs_scanvi = 20
n_hvgs          = 2000
batch_key       = None           # set to a column name to enable batch correction


In [2]:
# Parameters
reference_h5ad = "data/scvi/reference_su001.h5ad"
model_dir = "models/scanvi_reference_su001"
labels_key = "cluster"
n_epochs_scvi = 400
n_epochs_scanvi = 20


## 0 · Import

In [3]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import anndata as ad
import scanpy as sc
import scvi
import os, time, threading
import psutil, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

scvi.settings.seed = 42
t_nb_start = time.time()
_proc = psutil.Process()

def _mem_gb():
    return _proc.memory_info().rss / 1073741824

def _mps_mem_gb():
    try:
        if torch.backends.mps.is_available():
            return torch.mps.current_allocated_memory() / 1073741824
    except Exception:
        pass
    return 0.0

def _sys_free_gb():
    return psutil.virtual_memory().available / 1073741824

class ResourceMonitor:
    def __init__(self, interval=5, verbose=False):
        self.interval = interval
        self.verbose = verbose
        self.samples = []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def start(self):
        self._stop.clear()
        self._t0 = time.time()
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        self._thread.join(timeout=self.interval + 1)
        return self

    def _run(self):
        while not self._stop.wait(self.interval):
            s = {
                "t":        time.time(),
                "cpu":      _proc.cpu_percent(interval=None),
                "ram":      _mem_gb(),
                "sys_free": _sys_free_gb(),
                "mps":      _mps_mem_gb(),
            }
            self.samples.append(s)
            if self.verbose:
                import sys
                elapsed = s["t"] - self._t0
                print(
                    f"  [{elapsed:6.0f}s] CPU:{s['cpu']:5.0f}%  "
                    f"RAM:{s['ram']:5.2f}GB  Free:{s['sys_free']:5.2f}GB  "
                    f"MPS:{s['mps']:6.3f}GB",
                    flush=True
                )

    def summary(self, label=""):
        if not self.samples:
            return ("", "")
        cpus  = [s["cpu"]      for s in self.samples]
        rams  = [s["ram"]      for s in self.samples]
        frees = [s["sys_free"] for s in self.samples]
        mpss  = [s["mps"]      for s in self.samples]
        header = f"  {'':30s}  {'CPU%':>6}  {'RAM GB':>7}  {'SysFree':>8}  {'MPS GB':>7}"
        row    = (f"  {label:30s}  {max(cpus):6.0f}  {max(rams):7.2f}"
                  f"  {min(frees):8.2f}  {max(mpss):7.3f}")
        return header, row

print(f"scvi-tools {scvi.__version__} | scanpy {sc.__version__} | "
      f"torch {torch.__version__} | psutil {psutil.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"Initial RAM: {_mem_gb():.2f} GB | System free: {_sys_free_gb():.2f} GB")


Seed set to 42


scvi-tools 1.4.2 | scanpy 1.11.5 | torch 2.11.0 | psutil 7.2.2
MPS available: True
Initial RAM: 0.54 GB | System free: 2.95 GB


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_49504/467095701.py:84: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"scvi-tools {scvi.__version__} | scanpy {sc.__version__} | "


## 1 · Load Reference & Inspect Labels

In [4]:
adata = ad.read_h5ad(reference_h5ad)
print(f"Reference: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Layers   : {list(adata.layers.keys())}")

# Ensure labels_key exists; fall back to first string column
if labels_key not in adata.obs.columns:
    str_cols = [c for c in adata.obs.columns if adata.obs[c].dtype == object]
    labels_key = str_cols[0] if str_cols else None
    print(f"Warning: using '{labels_key}' as labels_key")

print(f"\nCell type distribution ('{labels_key}'):")
print(adata.obs[labels_key].value_counts().to_string())


Reference: 27,626 cells x 17,865 genes
Layers   : ['counts']

Cell type distribution ('cluster'):
cluster
Th17          6739
Naive         5001
CD8_mem       4736
CD8_act       3853
Tregs         3390
Tfh           1962
CD8_ex         902
CD8_eff        741
CD8_ex_act     302


## 2 · Preprocessing

Filter low-expressed genes and subset to highly variable genes.
`scVI` works on **raw integer counts** (stored in `layers['counts']`).

In [5]:
# ── Preprocessing: HVG selection ──────────────────────────────────────────────
# Store raw counts in a dedicated layer before normalisation
adata.layers["counts"] = adata.X.copy()

# Normalise + log1p on the .X slot so seurat-flavour HVG selection works
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Identify highly variable genes (seurat flavour is sparse-safe)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=n_hvgs,
    flavor="seurat",
    subset=True,
)
print(f"After HVG filter: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Layers available: {list(adata.layers.keys())}")


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


After HVG filter: 27,626 cells x 2,000 genes
Layers available: ['counts']


## 3 · Train SCVI

SCVI learns a low-dimensional latent space from raw counts using a variational autoencoder.
This is the backbone for SCANVI's semi-supervised label classifier.

In [6]:
print('=== Training SCVI ===')
t_scvi_start = time.time()
_mon_scvi = ResourceMonitor(interval=5, verbose=True).start()
print(f"  [  time] CPU%   RAM      Free     MPS")

scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key=batch_key)
scvi_model = scvi.model.SCVI(adata, n_latent=n_latent, n_layers=2, n_hidden=128)
print(scvi_model)

scvi_model.train(
    max_epochs=n_epochs_scvi,
    batch_size=512,
    early_stopping=True,
    early_stopping_patience=20,
    accelerator='mps',
)
_mon_scvi.stop()
t_scvi_elapsed = time.time() - t_scvi_start

epochs_run = len(scvi_model.history['elbo_train'])
print(f'\nSCVI: {epochs_run} epochs | {t_scvi_elapsed:.1f}s ({t_scvi_elapsed/60:.1f} min)')
_h, _r = _mon_scvi.summary('SCVI peak')
print(_h); print(_r)


=== Training SCVI ===
  [  time] CPU%   RAM      Free     MPS


SCVI model with the following parameters: 
n_hidden: 128, n_latent: 30, n_layers: 2, dropout_rate: 0.1, dispersion: gene, gene_likelihood: zinb, 
latent_distribution: normal.
Training status: Not Trained
Model's adata is minified?: False

/opt/anaconda3/envs/scrnaseq/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

  [   516s] CPU:   59%  RAM: 0.24GB  Free: 1.20GB  MPS: 0.022GB


  [   521s] CPU:   59%  RAM: 0.23GB  Free: 1.21GB  MPS: 0.038GB


  [   526s] CPU:   59%  RAM: 0.24GB  Free: 1.35GB  MPS: 0.022GB


`Trainer.fit` stopped: `max_epochs=400` reached.



SCVI: 400 epochs | 527.3s (8.8 min)
                                    CPU%   RAM GB   SysFree   MPS GB
  SCVI peak                           60     0.85      1.11    0.081


## 4 · Train SCANVI

SCANVI extends SCVI with a semi-supervised classifier. Cells with known
labels (`labels_key`) supervise the classifier; cells labeled `'Unknown'`
are treated as unlabelled and still contribute to the VAE.

In [7]:
print('=== Training SCANVI ===')
t_scanvi_start = time.time()
_mon_scanvi = ResourceMonitor(interval=5, verbose=True).start()
print(f"  [  time] CPU%   RAM      Free     MPS")

scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model,
    labels_key=labels_key,
    unlabeled_category='Unknown',
)
print(scanvi_model)

scanvi_model.train(
    max_epochs=n_epochs_scanvi,
    batch_size=512,
    n_samples_per_label=100,
    accelerator='mps',
)
_mon_scanvi.stop()
t_scanvi_elapsed = time.time() - t_scanvi_start

epochs_run = len(scanvi_model.history['elbo_train'])
print(f'\nSCANVI: {epochs_run} epochs | {t_scanvi_elapsed:.1f}s ({t_scanvi_elapsed/60:.1f} min)')
_h, _r = _mon_scanvi.summary('SCANVI peak')
print(_h); print(_r)


=== Training SCANVI ===
  [  time] CPU%   RAM      Free     MPS


ScanVI Model with the following params: 
unlabeled_category: Unknown, n_hidden: 128, n_latent: 30, n_layers: 2, dropout_rate: 0.1, dispersion: gene, 
gene_likelihood: zinb
Training status: Not Trained
Model's adata is minified?: False


INFO     Training for 20 epochs.                                                                                   


/opt/anaconda3/envs/scrnaseq/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been set to `mps`. Please note that not all PyTorch/Jax operations are supported with this backend. as a result, some models might be slower and less accurate than usual. Please verify your analysis!Refer to https://github.com/pytorch/pytorch/issues/77764 for more details.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores


...

  [    45s] CPU:   64%  RAM: 0.34GB  Free: 1.07GB  MPS: 0.187GB


  [    50s] CPU:   64%  RAM: 0.33GB  Free: 1.09GB  MPS: 0.175GB


  [    55s] CPU:   64%  RAM: 0.34GB  Free: 1.09GB  MPS: 0.061GB


`Trainer.fit` stopped: `max_epochs=20` reached.



SCANVI: 20 epochs | 56.8s (0.9 min)
                                    CPU%   RAM GB   SysFree   MPS GB
  SCANVI peak                         65     0.39      1.07    0.187


## 5 · Save Model

In [8]:
os.makedirs(model_dir, exist_ok=True)
# save_anndata=True stores the reference AnnData inside the model dir,
# so SCANVI.load(dir) works standalone on any machine.
scanvi_model.save(model_dir, overwrite=True, save_anndata=True)
print(f"SCANVI model saved -> {model_dir}/")
print(f"Contents: {os.listdir(model_dir)}")


SCANVI model saved -> models/scanvi_reference_su001/
Contents: ['model.pt', 'adata.h5ad']


## 6 · Reference UMAP in scANVI Latent Space

In [9]:
adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
sc.pp.neighbors(adata, use_rep="X_scANVI", n_neighbors=15)
sc.tl.umap(adata)

fig, ax = plt.subplots(figsize=(7, 3.5))
sc.pl.umap(adata, color=labels_key, title="Reference — scANVI latent space", ax=ax, show=False)
plt.tight_layout()
plt.savefig(os.path.join(model_dir, "reference_umap.png"), dpi=120)
plt.show()

# Save trained reference (with UMAP) for comparison plots later
out_trained = reference_h5ad.replace(".h5ad", "_trained.h5ad")
adata.write_h5ad(out_trained)
print(f"Saved trained reference -> {out_trained}")


Saved trained reference -> data/scvi/reference_su001_trained.h5ad


/var/folders/8y/ph922wl553g48t1wthjdsfnm0000gn/T/ipykernel_49504/3667338862.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7 · Timing Summary

In [10]:
t_total = time.time() - t_nb_start
all_samples = _mon_scvi.samples + _mon_scanvi.samples

print('\n' + '='*65)
print('  scvi/01 Resource & Timing Summary')
print('='*65)
print(f"  {'Step':<30}  {'Time (s)':>8}  {'Time (min)':>10}")
print(f"  {'-'*30}  {'-'*8}  {'-'*10}")
print(f"  {'SCVI training':<30}  {t_scvi_elapsed:8.1f}  {t_scvi_elapsed/60:10.1f}")
print(f"  {'SCANVI training':<30}  {t_scanvi_elapsed:8.1f}  {t_scanvi_elapsed/60:10.1f}")
print(f"  {'Total notebook':<30}  {t_total:8.1f}  {t_total/60:10.1f}")
if all_samples:
    peak_cpu = max(s['cpu']      for s in all_samples)
    peak_ram = max(s['ram']      for s in all_samples)
    min_free = min(s['sys_free'] for s in all_samples)
    peak_mps = max(s['mps']      for s in all_samples)
    print(f"\n  {'Resource':<30}  {'Peak/Min':>10}")
    print(f"  {'-'*30}  {'-'*10}")
    print(f"  {'CPU (% of 1 core)':<30}  {peak_cpu:>9.0f}%")
    print(f"  {'Process RAM peak':<30}  {peak_ram:>9.2f} GB")
    print(f"  {'System free RAM (min)':<30}  {min_free:>9.2f} GB")
    print(f"  {'MPS (GPU) memory peak':<30}  {peak_mps:>9.3f} GB")
print('='*65)
print(f'\nModel saved to: {model_dir}/')
print('Run scvi/02_project_query.ipynb to project new cells (no retraining).')



  scvi/01 Resource & Timing Summary
  Step                            Time (s)  Time (min)
  ------------------------------  --------  ----------
  SCVI training                      527.3         8.8
  SCANVI training                     56.8         0.9
  Total notebook                     621.0        10.3

  Resource                          Peak/Min
  ------------------------------  ----------
  CPU (% of 1 core)                      65%
  Process RAM peak                     0.85 GB
  System free RAM (min)                1.07 GB
  MPS (GPU) memory peak               0.187 GB

Model saved to: models/scanvi_reference_su001/
Run scvi/02_project_query.ipynb to project new cells (no retraining).
